# Synthetic Customer Generator

Generates a synthetic population of companies with **electricity** and, for a subset, **gas**
consumption, used to train the K-Means clustering model, plus a utility for generating a single
random test customer using the same generation logic - for exercising the app and pipeline
end-to-end with a known ground truth.

## Output

Written to `data/inputs/generated/`, in the canonical schema the rest of the pipeline expects:

```text
synthetic_metadata.csv            one row per customer (archetypes, targets, grid connection)
synthetic_electricity_long.csv    timestamp_local | timestamp_utc | customer_id | power_kw   | source_file
synthetic_gas_long.csv            timestamp_local | timestamp_utc | customer_id | energy_kwh | source_file   (has_gas customers only)
```

The single test customer is saved separately, under `data/inputs/generated/single_test/`, in the
same schema.


In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, "../src")
import synthetic_generator as sg
import time_cleaning as tc
import pandas as pd
import matplotlib.pyplot as plt

RANDOM_SEED = 42
YEAR = 2026
N_COMPANIES = 250

OUTPUT_DIR = Path("../data/inputs/generated")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SINGLE_TEST_DIR = OUTPUT_DIR / "single_test"
SINGLE_TEST_DIR.mkdir(parents=True, exist_ok=True)

print("Generator module loaded from:", sg.__file__)


## Bulk Generation

The population used to train the K-Means clustering model.

### Archetype reference

These live in `src/synthetic_generator.py` as the single source of truth - printed here only for
visibility, not redefined.

In [ ]:
print("Electricity archetypes:", list(sg.ELECTRICITY_ARCHETYPES))
print("Gas archetypes:", list(sg.GAS_ARCHETYPES))
print()
print("Electricity archetype -> gas coherence rule:")
display(pd.DataFrame(sg.ARCHETYPE_ENERGY_CONFIG).T)


### Generate the population

`generate_company_population()` handles archetype sampling, grid connection assignment, and the
electricity/gas coherence rule internally - see `src/synthetic_generator.py` for the exact logic.

In [ ]:
metadata, electricity_long, gas_long = sg.generate_company_population(
    n_companies=N_COMPANIES,
    year=YEAR,
    seed=RANDOM_SEED,
)

print("Companies:", len(metadata))
print("Electricity rows:", len(electricity_long))
print("Gas rows:", len(gas_long), f"({metadata['has_gas'].sum()} customers with gas)")
display(metadata.head())


### Validate the generated population

Checks:
- every electricity series covers a full calendar year at 15-minute resolution;
- every gas series (where present) covers a full calendar year at hourly resolution;
- annual totals match their targets;
- gas presence/archetype is coherent with the electricity archetype.

In [ ]:
# Row-count / resolution check
n_elec_timestamps = electricity_long["timestamp_utc"].nunique()
n_gas_timestamps = gas_long["timestamp_utc"].nunique() if not gas_long.empty else 0
expected_elec = len(pd.date_range(f"{YEAR}-01-01", f"{YEAR}-12-31 23:45", freq="15min"))
expected_gas = len(pd.date_range(f"{YEAR}-01-01", f"{YEAR}-12-31 23:00", freq="h"))

print(f"Electricity timestamps: {n_elec_timestamps} (expected {expected_elec})")
print(f"Gas timestamps: {n_gas_timestamps} (expected {expected_gas})")
assert n_elec_timestamps == expected_elec, "Electricity resolution/coverage mismatch"
if not gas_long.empty:
    assert n_gas_timestamps == expected_gas, "Gas resolution/coverage mismatch"


In [ ]:
# Annual total vs target check
elec_actual = (
    electricity_long.groupby("customer_id")["power_kw"].sum() * 0.25
).rename("actual_annual_electricity_kwh").reset_index()

check = metadata.merge(elec_actual, on="customer_id")
check["electricity_error_pct"] = (
    (check["actual_annual_electricity_kwh"] - check["target_annual_electricity_kwh"])
    / check["target_annual_electricity_kwh"] * 100
)
print("Max electricity annual error %:", check["electricity_error_pct"].abs().max())

if not gas_long.empty:
    gas_actual = gas_long.groupby("customer_id")["energy_kwh"].sum().rename(
        "actual_annual_gas_kwh"
    ).reset_index()
    gas_check = metadata[metadata["has_gas"]].merge(gas_actual, on="customer_id")
    gas_check["gas_error_pct"] = (
        (gas_check["actual_annual_gas_kwh"] - gas_check["target_annual_gas_kwh"])
        / gas_check["target_annual_gas_kwh"] * 100
    )
    print("Max gas annual error %:", gas_check["gas_error_pct"].abs().max())


In [ ]:
# Coherence check: gas presence/archetype should follow ARCHETYPE_ENERGY_CONFIG
display(
    metadata.groupby(["electricity_archetype", "has_gas", "gas_archetype"], dropna=False)
    .size()
    .rename("companies")
    .reset_index()
)


### DST cleanliness check

Confirms the population has no duplicate or missing timestamps around the EU daylight-saving
transitions.

In [ ]:
elec_dst_report = tc.validate_continuity(electricity_long, id_col="customer_id", expected_freq="15min")
assert (elec_dst_report["n_duplicate_utc_timestamps"] == 0).all(), "Duplicate timestamps in synthetic electricity!"
assert (elec_dst_report["n_missing_intervals"] == 0).all(), "Missing intervals in synthetic electricity!"
print("Electricity: no duplicate or missing timestamps across", len(elec_dst_report), "customers.")

if not gas_long.empty:
    gas_dst_report = tc.validate_continuity(gas_long, id_col="customer_id", expected_freq="h")
    assert (gas_dst_report["n_duplicate_utc_timestamps"] == 0).all(), "Duplicate timestamps in synthetic gas!"
    assert (gas_dst_report["n_missing_intervals"] == 0).all(), "Missing intervals in synthetic gas!"
    print("Gas: no duplicate or missing timestamps across", len(gas_dst_report), "customers.")


## Single Customer Generation for Testing Purposes

Generates **one fully-randomized company**, using the same generation logic as the bulk population
above - same archetype pool and weights, same electricity/gas coherence rule.

This is a test-data tool: it generates one customer's raw data, in the exact same schema as any
other customer (real or bulk-synthetic), with a **known archetype** printed below - so once this
customer runs through the rest of the pipeline, the assigned K-Means cluster can be checked against
that known archetype as a direct sanity check.

Re-run the generation cell as many times as you like; each run creates a new customer with its own
random `NRW_TEST_######` id, so nothing gets overwritten.

### Generate one fully-randomized customer

`seed=None` means a different random customer every time this cell runs. Set an explicit integer seed instead to reproduce the same test customer later.

In [ ]:
test_result = sg.generate_single_test_customer(year=YEAR, seed=None)

test_metadata = test_result["metadata"]
test_electricity = test_result["electricity"]
test_gas = test_result["gas"]

print("customer_id:           ", test_metadata["customer_id"])
print("electricity_archetype: ", test_metadata["electricity_archetype"])
print("gas_archetype:         ", test_metadata["gas_archetype"])
print("has_gas:               ", test_metadata["has_gas"])
print("target annual elec kWh:", round(test_metadata["target_annual_electricity_kwh"]))
if test_metadata["has_gas"]:
    print("target annual gas kWh: ", round(test_metadata["target_annual_gas_kwh"]))
print()
print(
    "Known archetype to check against once this customer runs through K-Means: ",
    test_metadata["electricity_archetype"],
    f"+ {test_metadata['gas_archetype']}" if test_metadata["has_gas"] else "(no gas)",
)


### Visual sanity check

Quick plot of the generated year - does the shape look like the intended archetype?

In [ ]:
fig, axes = plt.subplots(2 if test_metadata["has_gas"] else 1, 1, figsize=(14, 8), squeeze=False)

axes[0][0].plot(test_electricity["timestamp_local"], test_electricity["power_kw"], linewidth=0.6)
axes[0][0].set_title(f"Electricity — {test_metadata['customer_id']} ({test_metadata['electricity_archetype']})")
axes[0][0].set_ylabel("kW")
axes[0][0].grid(alpha=0.2)

if test_metadata["has_gas"]:
    axes[1][0].plot(test_gas["timestamp_local"], test_gas["energy_kwh"], linewidth=0.6, color="tab:orange")
    axes[1][0].set_title(f"Gas — {test_metadata['customer_id']} ({test_metadata['gas_archetype']})")
    axes[1][0].set_ylabel("kWh")
    axes[1][0].grid(alpha=0.2)

plt.tight_layout()
plt.show()


### DST cleanliness check

Same validator used for the bulk population above.

In [ ]:
test_elec_dst = tc.validate_continuity(test_electricity, id_col="customer_id", expected_freq="15min")
assert (test_elec_dst["n_duplicate_utc_timestamps"] == 0).all() and (test_elec_dst["n_missing_intervals"] == 0).all()
print("Electricity: clean.")

if test_gas is not None:
    test_gas_dst = tc.validate_continuity(test_gas, id_col="customer_id", expected_freq="h")
    assert (test_gas_dst["n_duplicate_utc_timestamps"] == 0).all() and (test_gas_dst["n_missing_intervals"] == 0).all()
    print("Gas: clean.")


## Save Outputs

In [ ]:
metadata_path = OUTPUT_DIR / "synthetic_metadata.csv"
electricity_path = OUTPUT_DIR / "synthetic_electricity_long.csv"
gas_path = OUTPUT_DIR / "synthetic_gas_long.csv"

metadata.to_csv(metadata_path, index=False)
electricity_long.to_csv(electricity_path, index=False)
gas_long.to_csv(gas_path, index=False)

print("Saved:")
print(" ", metadata_path)
print(" ", electricity_path)
print(" ", gas_path)


In [ ]:
test_customer_id = test_metadata["customer_id"]

pd.DataFrame([test_metadata]).to_csv(SINGLE_TEST_DIR / f"{test_customer_id}_metadata.csv", index=False)
test_electricity.to_csv(SINGLE_TEST_DIR / f"{test_customer_id}_electricity.csv", index=False)
if test_gas is not None:
    test_gas.to_csv(SINGLE_TEST_DIR / f"{test_customer_id}_gas.csv", index=False)

# Pointer file so downstream notebooks (02, 04, 05) know which single test
# customer to load, without guessing from file timestamps - keeps them all
# working on the SAME customer generated by this run, not silently drifting
# onto whichever file happens to be newest.
(SINGLE_TEST_DIR / "LATEST_CUSTOMER_ID.txt").write_text(test_customer_id)

print("Saved test customer:", test_customer_id, "->", SINGLE_TEST_DIR)
